# train.ipynb
This notebook contains the training code for a small language model using the TRL (Transformer Reinforcement Learning) library.

This notebook trains a tiny GPT-2 style language model using the specified tokenizer architecture and data.


In [ ]:
# ============================================================
# Colab Setup Cell
# ============================================================
!pip install -q transformers trl datasets sentencepiece bitsandbytes accelerate
from google.colab import files
import zipfile
import os
print("Please upload the data zip file.")
uploaded = files.upload()
zip_file_name = next((fn for fn in uploaded if fn.endswith('.zip')), None)
if zip_file_name:
    print(f"Extracting data from {zip_file_name}...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Data extracted.")
else:
    print("ERROR: No zip file found.")
os.chdir('/content')


## Installation
First, we need to install the required dependencies for training:


In [ ]:
!pip install transformers trl bitsandbytes

In [ ]:
from transformers import set_seed
from transformers import AutoConfig, AutoModelForCausalLM, DebertaV2Tokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import os
def preprocess_logits_for_metrics(logits, labels):
    """Extract predicted token IDs from model logits for evaluation"""
    pred_ids = torch.argmax(logits, dim=-1)  # Get the token with highest probability
    return pred_ids
def compute_metrics(eval_pred):
    """Calculate accuracy by comparing predictions with true labels"""
    logits, labels = eval_pred
    predictions = logits.flatten()  # Flatten to 1D array
    labels = labels.flatten()
    # Only consider non-padding tokens (labels != -100 are actual tokens)
    mask = labels != -100
    labels = labels[mask]
    predictions = predictions[mask]
    # Calculate accuracy
    correct = labels == predictions
    accuracy = correct.sum() / float(len(correct))
    return {"acc": accuracy}

In [ ]:
# Load the custom tokenizer trained on BabyLM dataset
model_name = "openai-community/gpt2"
TOKENIZER_PATH = '/content/tokenizer_dir/tokenizer.model'
tokenizer = DebertaV2Tokenizer(TOKENIZER_PATH)
# Fix vocabulary size to include special tokens
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
# Check for special token IDs
special_token_ids = [
    tokenizer.pad_token_id,
    tokenizer.unk_token_id,
    tokenizer.bos_token_id,
    tokenizer.eos_token_id
]
# Filter out None values before finding max
filtered_special_token_ids = [id for id in special_token_ids if id is not None]
max_token_id = max(filtered_special_token_ids) if filtered_special_token_ids else -1 # Handle case where all are None
print(f"Max special token ID: {max_token_id}")
# Ensure vocab size covers all tokens including specials
required_vocab_size = max(tokenizer.vocab_size, max_token_id + 1)
print(f"Required vocab size: {required_vocab_size}")
# Create custom configuration based on GPT-2 but with smaller dimensions
config = AutoConfig.from_pretrained(model_name)
config.hidden_size = 384 # Same dimensionality as the best performing BabyLM model.
config.intermediate_size = 1280  # Feed-forward intermediate size
config.vocab_size = required_vocab_size  # Use required size to include special tokens
# Initialize model with custom configuration
model = AutoModelForCausalLM.from_config(config)
# Print model size for reference, should be around 31M parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of model parameters: {num_params}")
# Load training and validation datasets
TRAIN_DATA = '/content/train.txt'
VAL_DATA = '/content/dev.txt'
dataset = load_dataset('text', data_files={'train': TRAIN_DATA, 'validation': VAL_DATA})
print(dataset)
# Tokenize the dataset
max_length = 64 # Max sequence length from original SFTConfig
def tokenize_function(examples):
    # Ensure all examples are strings, and handle potential non-string entries
    texts = [str(t) if t is not None else "" for t in examples['text']]
    return tokenizer(texts, truncation=True, max_length=max_length)
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"], # Remove original text column
    num_proc=1 # Using 1 process for simplicity and avoiding potential multiprocessing issues in Colab
)
print("Tokenized dataset:")
print(tokenized_dataset)

In [ ]:
# Fix vocabulary size to include special tokens
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
# Check for special token IDs
special_token_ids = [
    tokenizer.pad_token_id,
    tokenizer.unk_token_id,
    tokenizer.bos_token_id,
    tokenizer.eos_token_id
]
max_token_id = max([id for id in special_token_ids if id is not None])
print(f"Max special token ID: {max_token_id}")
# Ensure vocab size covers all tokens including specials
required_vocab_size = max(tokenizer.vocab_size, max_token_id + 1)
print(f"Required vocab size: {required_vocab_size}")

In [ ]:
# Set random seed for reproducibility
set_seed(0)
# Custom data collator that works with SentencePiece
def custom_data_collator(features):
    # Get max length in this batch
    max_length = max(len(f['input_ids']) for f in features)
    # Pad sequences
    batch = {
        'input_ids': [],
        'labels': [],
        'attention_mask': []
    }
    # SentencePiece default pad token id is 0. Use tokenizer's pad_token_id if available.
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    for f in features:
        seq = f['input_ids']
        seq_len = len(seq)
        # Pad to max_length
        padding_length = max_length - seq_len
        padded_seq = seq + [pad_token_id] * padding_length
        attention = [1] * seq_len + [0] * padding_length
        batch['input_ids'].append(padded_seq)
        batch['labels'].append(padded_seq)  # For LM, labels = inputs
        batch['attention_mask'].append(attention)
    # Convert to tensors
    return {
        'input_ids': torch.tensor(batch['input_ids'], dtype=torch.long),
        'labels': torch.tensor(batch['labels'], dtype=torch.long),
        'attention_mask': torch.tensor(batch['attention_mask'], dtype=torch.long)
    }
# Training arguments
training_args = TrainingArguments(
    output_dir="outputs/model_output",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=custom_data_collator,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
)
# Start training
trainer.train()

In [ ]:
# Save model and tokenizer to Google Drive
print("\n" + "="*70)
print("Saving model and tokenizer to Google Drive...")
print("="*70)
final_output_dir = '/content/drive/MyDrive/babylm_models/model_output'
import os
os.makedirs(final_output_dir, exist_ok=True)
# Save model
trainer.save_model(final_output_dir)
print(f"Model saved to: {final_output_dir}")
# Save tokenizer
tokenizer.save_pretrained(final_output_dir)
print(f"Tokenizer saved to: {final_output_dir}")
# Save training config
import json
config_info = {
    'model_type': 'GPT-2',
    'tokenizer': TOKENIZER_PATH,
    'hidden_size': config.hidden_size,
    'vocab_size': config.vocab_size,
    'num_parameters': sum(p.numel() for p in model.parameters()),
    'epochs': 10,
}
with open(os.path.join(final_output_dir, 'training_info.json'), 'w') as f:
    json.dump(config_info, f, indent=2)
print(f"Training info saved")
print("\n" + "="*70)
print("Training complete. Model saved.")
print("="*70)
print(f"\nYou can find your model at: {final_output_dir}")
print("This will persist even after the Colab session ends.")

In [ ]:
import shutil
from google.colab import files
import os
# Define the directory to be zipped (from previous cell output)
final_output_dir = '/content/drive/MyDrive/babylm_models/model_output'
# Define the name for the zip file
zip_file_name = 'model_output.zip'
output_archive = '/content/' + zip_file_name.replace('.zip', '') # shutil will add .zip
print(f"Compressing '{final_output_dir}' into '{zip_file_name}'...")
# Create the zip archive
shutil.make_archive(output_archive, 'zip', final_output_dir)
print(f"Successfully created '{zip_file_name}'")
# Offer the zip file for download
print(f"Initiating download for '{zip_file_name}'...")
files.download(output_archive + '.zip')
print("Download initiated.")